In [1]:
# ============================================================
# Load, preprocess, and experiment with dataset
# ============================================================

# Install if needed:
# pip install pandas numpy scikit-learn openpyxl matplotlib

import pandas as pd
import numpy as np
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


C:\Users\Beta\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [3]:
pip install openpyxl==3.1.2

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\Beta\\anaconda3\\Lib\\site-packages\\~penpyxl\\utils\\cell.cp310-win_amd64.pyd'
Consider using the `--user` option or check the permissions.




     ------------------------------------ 250.0/250.0 kB 426.6 kB/s eta 0:00:00
  Attempting uninstall: openpyxl
    Found existing installation: openpyxl 3.0.10
    Uninstalling openpyxl-3.0.10:
      Successfully uninstalled openpyxl-3.0.10


In [10]:
# ============================================================
# 1. LOAD DATASET
# ============================================================

# GitHub raw file URL
url = "https://raw.githubusercontent.com/uoldexplain/Dataset/main/UOLD-Explain.xlsx"

# Read Excel file
df = pd.read_excel(url)

print("Dataset Loaded Successfully")
print("-" * 50)

# Display first rows
print(df.head())

# Dataset info
print("\nDataset Shape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

# ============================================================
# 2. HANDLE MISSING VALUES
# ============================================================

print("\nMissing Values:")
print(df.isnull().sum())

# Example column names
# Replace these with actual dataset columns
TEXT_COLUMN = "Tweet_privacy_Protected"
LABEL_COLUMN = "Label"

# Remove rows with missing text/label
df = df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN])

print("\nShape After Removing Missing Values:")
print(df.shape)

Dataset Loaded Successfully
--------------------------------------------------
   Unnamed: 0  Sr.No  Tweet_Id  \
0           0      0         2   
1           1      1       111   
2           2      2       115   
3           3      3       122   
4           4      4       148   

                             Tweet_privacy_Protected  \
0  USER دو بدنسلئیے ، حرام خور منافق مل رہے ہیں پ...   
1  شاہ سلمان کی دعوت پر چین کے صدر کل سعودی عرب پ...   
2  USER لوگ تو پاکستانی ہیں یہ گھٹیا کیسے ہو گ۶ے ...   
3                   USER گھٹیا انسان دنیا ہی چھوڑ دو   
4  Emoji Emoji Emoji Emoji Emoji Emoji *{مرد کیلئ...   

                                    Rationale_Vector  \
0                    0 0 1 0 1 1 1 0 0 0 0 0 0 0 0 0   
1                    1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1   
2  1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 ...   
3                                      1 1 1 0 0 0 0   
4  1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 ...   

                               Annotators 

In [11]:
# ============================================================
# 3. TEXT PREPROCESSING
# ============================================================
def preprocess_text(text):
    # Convert to string
    text = str(text)
    # Lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)
    # Remove mentions and hashtags
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    # Remove numbers
    text = re.sub(r"\d+", "", text)
    # Remove punctuation
    text = text.translate(
        str.maketrans("", "", string.punctuation))
    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    return text


# Apply preprocessing
df["clean_text"] = df[TEXT_COLUMN].apply(preprocess_text)
print("\nSample Cleaned Text:")
print(df[[TEXT_COLUMN, "clean_text"]].head())

# ============================================================
# 4. LABEL DISTRIBUTION
# ============================================================

print("\nLabel Distribution:")
print(df[LABEL_COLUMN].value_counts())


Sample Cleaned Text:
                             Tweet_privacy_Protected  \
0  USER دو بدنسلئیے ، حرام خور منافق مل رہے ہیں پ...   
1  شاہ سلمان کی دعوت پر چین کے صدر کل سعودی عرب پ...   
2  USER لوگ تو پاکستانی ہیں یہ گھٹیا کیسے ہو گ۶ے ...   
3                   USER گھٹیا انسان دنیا ہی چھوڑ دو   
4  Emoji Emoji Emoji Emoji Emoji Emoji *{مرد کیلئ...   

                                          clean_text  
0  user دو بدنسلئیے ، حرام خور منافق مل رہے ہیں پ...  
1  شاہ سلمان کی دعوت پر چین کے صدر کل سعودی عرب پ...  
2  user لوگ تو پاکستانی ہیں یہ گھٹیا کیسے ہو گے ا...  
3                   user گھٹیا انسان دنیا ہی چھوڑ دو  
4  emoji emoji emoji emoji emoji emoji مرد کیلئے ...  

Label Distribution:
Label
0    15404
1     4823
3     2105
2     1168
4      563
Name: count, dtype: int64


In [12]:
#Multi-class classification

# ============================================================
# 5. TRAIN TEST SPLIT
# ============================================================

X = df["clean_text"]
y = df[LABEL_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining Samples:", len(X_train))
print("Testing Samples:", len(X_test))

# ============================================================
# 6. EXPERIMENT 1 - LOGISTIC REGRESSION
# ============================================================

lr_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=5000,
        ngram_range=(1,2)
    )),
    ("clf", LogisticRegression(max_iter=1000))
])

# Train model
lr_pipeline.fit(X_train, y_train)

# Predictions
y_pred_lr = lr_pipeline.predict(X_test)

print("\n" + "=" * 50)
print("LOGISTIC REGRESSION RESULTS")
print("=" * 50)

print("\nAccuracy:")
print(accuracy_score(y_test, y_pred_lr))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))

# ============================================================
# 7. EXPERIMENT 2 - NAIVE BAYES
# ============================================================

nb_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=5000,
        ngram_range=(1,2)
    )),
    ("clf", MultinomialNB())
])

# Train model
nb_pipeline.fit(X_train, y_train)

# Predictions
y_pred_nb = nb_pipeline.predict(X_test)

print("\n" + "=" * 50)
print("NAIVE BAYES RESULTS")
print("=" * 50)

print("\nAccuracy:")
print(accuracy_score(y_test, y_pred_nb))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_nb))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb))



Training Samples: 19250
Testing Samples: 4813

LOGISTIC REGRESSION RESULTS

Accuracy:
0.7554539788073966

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.97      0.86      3081
           1       0.69      0.53      0.60       965
           2       0.65      0.24      0.35       234
           3       0.63      0.17      0.27       421
           4       0.59      0.09      0.16       112

    accuracy                           0.76      4813
   macro avg       0.67      0.40      0.45      4813
weighted avg       0.73      0.76      0.72      4813


Confusion Matrix:
[[2984   75    9    8    5]
 [ 419  515   12   19    0]
 [ 128   37   56   12    1]
 [ 238  106    5   71    1]
 [  79   16    4    3   10]]

NAIVE BAYES RESULTS

Accuracy:
0.7224184500311656

Classification Report:
              precision    recall  f1-score   support

           0       0.74      0.98      0.84      3081
           1       0.63      0.44    

C:\Users\Beta\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\Beta\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
C:\Users\Beta\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [13]:
# Original class distribution
print("Original Distribution:")
print(df[LABEL_COLUMN].value_counts())

# Combine classes 1,2,3,4 into class 1
df[LABEL_COLUMN] = df[LABEL_COLUMN].replace({
    2: 1,
    3: 1,
    4: 1
})

# New class distribution
print("\nNew Distribution:")
print(df[LABEL_COLUMN].value_counts())

Original Distribution:
Label
0    15404
1     4823
3     2105
2     1168
4      563
Name: count, dtype: int64

New Distribution:
Label
0    15404
1     8659
Name: count, dtype: int64


In [14]:
#Binary classification

# ============================================================
# 5. TRAIN TEST SPLIT
# ============================================================

X = df["clean_text"]
y = df[LABEL_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining Samples:", len(X_train))
print("Testing Samples:", len(X_test))

# ============================================================
# 6. EXPERIMENT 1 - LOGISTIC REGRESSION
# ============================================================

lr_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=5000,
        ngram_range=(1,2)
    )),
    ("clf", LogisticRegression(max_iter=1000))
])

# Train model
lr_pipeline.fit(X_train, y_train)

# Predictions
y_pred_lr = lr_pipeline.predict(X_test)

print("\n" + "=" * 50)
print("LOGISTIC REGRESSION RESULTS")
print("=" * 50)

print("\nAccuracy:")
print(accuracy_score(y_test, y_pred_lr))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))

# ============================================================
# 7. EXPERIMENT 2 - NAIVE BAYES
# ============================================================

nb_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=5000,
        ngram_range=(1,2)
    )),
    ("clf", MultinomialNB())
])

# Train model
nb_pipeline.fit(X_train, y_train)

# Predictions
y_pred_nb = nb_pipeline.predict(X_test)

print("\n" + "=" * 50)
print("NAIVE BAYES RESULTS")
print("=" * 50)

print("\nAccuracy:")
print(accuracy_score(y_test, y_pred_nb))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_nb))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb))



Training Samples: 19250
Testing Samples: 4813

LOGISTIC REGRESSION RESULTS

Accuracy:
0.823187201329732

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.94      0.87      3081
           1       0.85      0.62      0.72      1732

    accuracy                           0.82      4813
   macro avg       0.83      0.78      0.79      4813
weighted avg       0.83      0.82      0.82      4813


Confusion Matrix:
[[2892  189]
 [ 662 1070]]

NAIVE BAYES RESULTS

Accuracy:
0.8204861832536879

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.95      0.87      3081
           1       0.87      0.59      0.70      1732

    accuracy                           0.82      4813
   macro avg       0.84      0.77      0.79      4813
weighted avg       0.83      0.82      0.81      4813


Confusion Matrix:
[[2933  148]
 [ 716 1016]]
